# HealthGraph — ETL
Matrizes curriculares da UFF (PDF) → pandas → Neo4j.
Estrutura baseada na **Aula 10 — Bancos de Dados de Grafos: Neo4j** (TCC00336).

| Etapa | O que faz |
|---|---|
| **0. Modelagem** | Do relacional ao grafo, convenções de nomes |
| **1. Extract** | Ler os PDFs de `assets/matrizes/` |
| **2. Transform** | Uma tabela pandas por elemento do grafo |
| **3. Load** | Schema (`cypher/schema.cypher`) + `MERGE` com parâmetros |
| **4. Validação** | Contagens e consultas (`cypher/queries.cypher`) |

## 0. Modelagem
**Do modelo relacional ao grafo** (slide 38):

| Relacional / ER | Grafo (Neo4j) |
|---|---|
| Tabela / entidade | Label de nó |
| Linha (instância) | Nó |
| Coluna | Propriedade |
| Chave estrangeira | Relacionamento |
| Tabela associativa (N:N) | Relacionamento, com os atributos da associação como propriedades |
| Chave primária | Propriedade identificadora com constraint de unicidade |

**Convenções (slide 15):** labels em `PascalCase`, relacionamentos em `MAIÚSCULAS_COM_SUBLINHADO` (verbos que leem bem na direção da seta), propriedades em `camelCase`.

**Boas práticas (slide 40):** partir das perguntas, relacionamentos com verbo e direção, promover propriedade a nó quando for preciso navegar por ela, índices nos pontos de entrada.

**Perguntas que o grafo deve responder:**
- TODO: quais disciplinas os cursos dividem?
- TODO: quais disciplinas travam o andamento da grade?

**Modelo (rascunho):**
```
TODO: definir nós, relacionamentos e propriedades
```

## 1. Extract

In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
import pdfplumber
from neo4j import GraphDatabase

NEO4J_URI  = os.getenv("NEO4J_URI", "neo4j://localhost:7687")
NEO4J_AUTH = (os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", "neo4j12345"))
NEO4J_DB   = os.getenv("NEO4J_DATABASE", "neo4j")

PDF_DIR    = Path("assets/matrizes")
CYPHER_DIR = Path("cypher")

driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
driver.verify_connectivity()

In [ ]:
# TODO: ler cada PDF de PDF_DIR e extrair cabeçalho do curso, disciplinas e requisitos

## 2. Transform

In [ ]:
# TODO: montar um DataFrame por elemento do grafo (nós e relacionamentos)
# TODO: converter NaN em None antes de enviar ao Neo4j

## 3. Load
1. **Schema:** executar `cypher/schema.cypher` (constraints e índices).
2. **Dados:** `MERGE` é idempotente (slide 30); valores sempre por parâmetro, nunca por concatenação (slide 42); `UNWIND` em lotes.

In [ ]:
def executar_arquivo_cypher(caminho):
    texto = re.sub(r"//.*", "", Path(caminho).read_text(encoding="utf-8"))
    comandos = [c.strip() for c in texto.split(";") if c.strip()]
    for c in comandos:
        driver.execute_query(c, database_=NEO4J_DB)
    return len(comandos)

# executar_arquivo_cypher(CYPHER_DIR / "schema.cypher")

In [ ]:
# TODO: queries MERGE e carga em lotes
# Exemplo de formato:
# driver.execute_query("UNWIND $rows AS r MERGE (...)", rows=registros, database_=NEO4J_DB)

## 4. Validação

In [ ]:
# TODO: contagem de nós e relacionamentos; consultas de cypher/queries.cypher

In [ ]:
driver.close()